In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd

df = pd.read_csv(str(DATA_DIR / 'General Lexicon DHH Annotations.tsv'), sep="\t")

print(df.columns)
print(df.head(5))

df["word"] = df["word"].astype(str).str.strip()


In [ ]:
# === STEP 1: Clean DHH TSV → final per-word complexity scores, plus SFT format helpers ===
import pandas as pd
import numpy as np
import json
from pathlib import Path
import pickle

# DATA_DIR is configured in the first cell.
TSV_PATH = DATA_DIR / 'General Lexicon DHH Annotations.tsv'
OUT_DIR = PROJECT_ROOT / "prepared_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Load & basic hygiene
df = pd.read_csv(TSV_PATH, sep="\t")
df["word"] = df["word"].astype(str).str.strip()

# 2) Force annotator cols to numeric and treat -1 as missing
annotator_cols = [c for c in df.columns if c.lower().startswith("annotator")]
for c in annotator_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")  # bad strings -> NaN
df[annotator_cols] = df[annotator_cols].replace(-1, np.nan)

# Ensure 'average' is numeric if present
if "average" in df.columns:
    df["average"] = pd.to_numeric(df["average"], errors="coerce")

# Recompute a clean mean and pick final score
df["dhh_mean_no_minus1"] = df[annotator_cols].mean(axis=1) if annotator_cols else np.nan
df["final_score"] = df["dhh_mean_no_minus1"].fillna(df.get("average")).astype(float)

# Create a Python dictionary mapping each lowercase word to its final score
complexity_dict = dict(zip(df["word"].str.lower(), df["final_score"]))

# Save a clean lexicon and the dict for fast loading later
df_out = df[["word", "final_score", "average"] + annotator_cols].copy()
df_out.to_csv(OUT_DIR / 'dhh_lexicon_clean.csv', index=False)
with open(OUT_DIR / "complexity_dict.pkl", "wb") as f:
    pickle.dump(complexity_dict, f)

print(f"Saved clean lexicon → {OUT_DIR / 'dhh_lexicon_clean.csv'}")
print(f"Saved complexity dict → {OUT_DIR/'complexity_dict.pkl'}")
print("Sample scores:", {w: complexity_dict.get(w) for w in ["a", "aa", "aaliyah", "aaron", "abandon"]})



In [ ]:
for w in complexity_dict:
    print(w, "=", complexity_dict.get(w))

In [ ]:
# === STEP 2: Auto-generate (instruction, input, output) pairs using DHH scores + WordNet ===
import sys, subprocess, os, re, json, pickle
from pathlib import Path

# 0) Ensure complexity_dict is loaded from your cleaned step
OUT_DIR = PROJECT_ROOT / "prepared_data"
PKL_PATH = OUT_DIR / "complexity_dict.pkl"

if "complexity_dict" not in globals():
    with open(PKL_PATH, "rb") as f:
        complexity_dict = pickle.load(f)
print(f"Loaded {len(complexity_dict)} words from {PKL_PATH}")

def dhh_score(token: str, default: float = 10.0) -> float:
    return complexity_dict.get(token.lower(), default)

# 1) Install and import NLTK + WordNet if needed
def ensure_package(pkg: str):
    try:
        __import__(pkg)
    except ModuleNotFoundError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

ensure_package("nltk")
import nltk
try:
    from nltk.corpus import wordnet as wn
    _ = wn.synsets("test")
except LookupError:
    print("Downloading WordNet data ...")
    nltk.download("wordnet")
    nltk.download("omw-1.4")
    from nltk.corpus import wordnet as wn

# 2) Simplification helpers (regex tokenizer + synonym search guided by DHH)
_word_pat = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def find_simpler_synonym(word: str, min_delta: float = 0.3) -> str | None:
    """
    Return a synonym whose DHH score is lower than the original by at least `min_delta`.
    Lower DHH score = simpler. None if no better synonym found.
    """
    base = dhh_score(word)
    best = None
    best_score = base
    for syn in wn.synsets(word):
        for lemma in syn.lemmas():
            cand = lemma.name().replace("_", " ")
            if cand.lower() == word.lower():
                continue
            s = dhh_score(cand)
            if s + 1e-9 < best_score - min_delta:
                best, best_score = cand, s
    return best

def simplify_sentence_lexical(sentence: str, threshold: float = 4.5, min_delta: float = 0.3) -> str:
    """
    Replace words whose DHH score >= threshold with simpler synonyms if available.
    Preserves punctuation spacing reasonably well.
    """
    tokens = _word_pat.findall(sentence)
    out = []
    for tok in tokens:
        if tok.isalpha():
            base = dhh_score(tok)
            if base >= threshold:
                repl = find_simpler_synonym(tok, min_delta=min_delta)
                out.append(repl if repl else tok)
            else:
                out.append(tok)
        else:
            out.append(tok)
    # Rejoin with minimal spacing rules
    result = []
    for i, t in enumerate(out):
        if i > 0 and (t.isalnum() and result[-1][-1].isalnum()):
            result.append(" ")
        result.append(t)
    return "".join(result)

# 3) Build SFT examples (instruction/input/output)
SFT_INSTRUCTION = "Simplify the following sentence using simpler language."

def to_sft(original: str, simplified: str) -> dict:
    return {"instruction": SFT_INSTRUCTION, "input": original, "output": simplified}

def make_sft_examples(sentences: list[str],
                      threshold: float = 4.5,
                      min_delta: float = 0.3,
                      drop_if_unchanged: bool = True) -> list[dict]:
    examples = []
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        simplified = simplify_sentence_lexical(s, threshold=threshold, min_delta=min_delta)
        if drop_if_unchanged and simplified.strip() == s.strip():
            continue
        examples.append(to_sft(s, simplified))
    return examples

def write_jsonl(examples: list[dict], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"Wrote {len(examples)} examples → {path}")

# 4) Provide sentences:
# Option A: read from a text file (one sentence per line) if present
RAW_PATH = PROJECT_ROOT / "prepared_data" / "raw_sentences.txt"
if RAW_PATH.exists():
    with RAW_PATH.open("r", encoding="utf-8") as f:
        raw_sentences = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(raw_sentences)} sentences from {RAW_PATH}")
else:
    # Option B: small demo list (you can expand later)
    raw_sentences = [
        "The committee utilized a plethora of data to reach its conclusion.",
        "The professor elucidated the enigmatic concept during the lecture.",
        "He commenced the procedure subsequent to the preliminary assessment.",
        "Her philanthropic contributions ameliorated conditions within the community.",
        "They attempted to obfuscate the salient details of the agreement."
    ]
    print(f"No {RAW_PATH} found; using {len(raw_sentences)} demo sentences.")

# 5) Generate and save training JSONL
train_examples = make_sft_examples(raw_sentences, threshold=4.2, min_delta=0.2, drop_if_unchanged=True)
WRITE_PATH = OUT_DIR / "train_auto_sft.jsonl"
write_jsonl(train_examples, WRITE_PATH)

# Show a couple of examples
if train_examples:
    print("\nExample 1:", train_examples[0])
    if len(train_examples) > 1:
        print("Example 2:", train_examples[1])
else:
    print("No examples were generated. Try lowering 'threshold' or 'min_delta'.")


In [ ]:
# === Check column names and first sample ===
from datasets import load_dataset

dataset = load_dataset("bogdancazan/wikilarge-text-simplification")
print(dataset)

# show column names of one split
print("\nColumn names:", dataset["train"].column_names)

# show the first row to identify complex/simple fields
print("\nSample row:\n", dataset["train"][0])


In [ ]:
# Columns for this dataset
complex_col = "Normal"
simple_col  = "Simple"

# Quick sanity check
for i in range(3):
    print(f"\nComplex: {dataset['train'][i][complex_col]}")
    print(f"Simple:  {dataset['train'][i][simple_col]}")


In [ ]:
!pip install -q transformers datasets sentencepiece accelerate

from datasets import DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, TrainingArguments, Trainer
)
import torch

model_name = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

max_input_len  = 256
max_target_len = 256

def preprocess(batch):
    # encoder inputs (complex)
    model_inputs = tokenizer(
        batch[complex_col],
        max_length=max_input_len,
        truncation=True,
        padding=False,              # let collator pad dynamically
    )
    # decoder targets (simple)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch[simple_col],
            max_length=max_target_len,
            truncation=True,
            padding=False
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize all splits that exist
tokenized = {}
for split in dataset.keys():
    tokenized[split] = dataset[split].map(
        preprocess,
        batched=True,
        remove_columns=dataset[split].column_names,
        desc=f"Tokenizing {split}"
    )
tokenized_ds = DatasetDict(tokenized)

# Dynamic padding + correct label masking
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Basic training config (adjust batch sizes to your GPU)
args = TrainingArguments(
    output_dir="bart-wikilarge-simplifier",
    learning_rate=5e-5,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)


trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    data_collator=collator,
    train_dataset=tokenized_ds.get("train"),
    eval_dataset=tokenized_ds.get("validation"),
)

trainer.train()
trainer.evaluate()


# Quick smoke test
def generate_simple(s, max_new_tokens=96):
    inp = tokenizer(s, return_tensors="pt", truncation=True, max_length=max_input_len).to(model.device)
    gen = model.generate(**inp, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(gen[0], skip_special_tokens=True)

test_src = dataset["test"][0][complex_col]
print("\n--- SAMPLE ---")
print("INPUT:\n", test_src)
print("OUTPUT:\n", generate_simple(test_src))


In [ ]:
# === STEP 3: Fine-tune BART on your simplification pairs (JSONL) ===
# Uses: prepared_data/train_auto_sft.jsonl with fields: instruction, input, output

!pip install -q transformers datasets accelerate sentencepiece

import json
from pathlib import Path
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, TrainingArguments, Trainer
)

TRAIN_DATA_DIR = PROJECT_ROOT / "prepared_data"
TRAIN_PATH = TRAIN_DATA_DIR / "train_auto_sft.jsonl"
assert TRAIN_PATH.exists(), f"Missing {TRAIN_PATH}. Generate it in the previous step."

# 1) Load JSONL into a HF dataset
raw_ds = load_dataset("json", data_files={"train": str(TRAIN_PATH)})

# (Optional) create a tiny validation split from train for monitoring
raw_ds = raw_ds["train"].train_test_split(test_size=0.05, seed=42)
ds = DatasetDict({"train": raw_ds["train"], "validation": raw_ds["test"]})

# 2) Choose a BART checkpoint (base to start; you can try 'facebook/bart-large' later)
model_name = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3) Build source/target text from your SFT schema
# We'll unify the format as:  source = "<instruction>\n\n<input>", target = "<output>"
INSTRUCTION_FALLBACK = "Simplify the following sentence using simpler language."

def build_source(ex):
    instr = (ex.get("instruction") or INSTRUCTION_FALLBACK).strip()
    src = (ex.get("input") or "").strip()
    return f"{instr}\n\n{src}"

def build_target(ex):
    return (ex.get("output") or "").strip()

# 4) Tokenize with sensible lengths for BART
max_source_len = 256
max_target_len = 256

def preprocess(batch):
    sources = [build_source(ex) for ex in batch]
    targets = [build_target(ex) for ex in batch]
    model_inputs = tokenizer(
        sources, max_length=max_source_len, truncation=True, padding=False
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets, max_length=max_target_len, truncation=True, padding=False
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = ds.map(
    preprocess,
    batched=True,
    remove_columns=ds["train"].column_names,
    desc="Tokenizing"
)

# 5) Data collator for seq2seq (handles dynamic padding + label masking)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 6) Training args (tweak for your GPU)
args = TrainingArguments(
    output_dir="bart-simplifier",
    learning_rate=5e-5,
    num_train_epochs=2,               # start small; increase if needed
    per_device_train_batch_size=8,    # reduce if OOM; try 4 or 2
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,    # increase if you need larger effective batch
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True,                        # if your GPU supports it
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    data_collator=collator,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
)

trainer.train()

# 7) Quick smoke test: generate on a few examples
def generate(text, max_new_tokens=64):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_source_len).to(model.device)
    gen = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(gen[0], skip_special_tokens=True)

test_src = "Simplify the following sentence using simpler language.\n\nThe committee utilized a plethora of data to reach its conclusion."
print("\n--- SAMPLE ---")
print("INPUT:\n", test_src)
print("OUTPUT:\n", generate(test_src))
